# Attestor 4.3 — QLoRA Fine-Tune on Kaggle (v3: 1 epoch, SSTI fix, longer outputs)
# Base model: Qwythos-9B (Mythos-distilled) on T4+, Dolphin Mistral 7B fallback on P100.
# Training data: 3,790 Attestor security examples (47 CWE-94/SSTI examples to fix SSRF hallucination).
# v3 changes: 1 epoch (fix overfitting), 39 new SSTI examples (avg 1430 chars), balanced CWE-94/CWE-918 ratio.
# **Runtime**: GPU T4 x2 (recommended) or P100. Enable GPU in Settings > Accelerator.

In [ ]:
# Cell 1: Detect GPU and install compatible dependencies
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader,nounits'],
                       capture_output=True, text=True)
gpu_cap = result.stdout.strip().split('\n')[0]
print(f'GPU compute capability: {gpu_cap}')

P100_MODE = gpu_cap.startswith('6.')

# Remove torchvision — text-only fine-tuning doesn't need it, and it causes
# circular import errors when torch version is changed.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'],
               capture_output=True)
# Also remove torchao which crashes on some Kaggle envs
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
               capture_output=True)

# Step 1: Install transformers from git source (required for Qwen 3.5 / Qwythos-9B)
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/huggingface/transformers.git',
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'peft', 'trl',
    'accelerate', 'datasets', 'bitsandbytes',
    'sentencepiece', 'protobuf',
])

# Step 2: Try installing Gated DeltaNet kernels (needed for Qwythos performance)
try:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'flash-linear-attention', 'causal-conv1d',
    ])
    print('DeltaNet kernels installed.')
except subprocess.CalledProcessError:
    print('DeltaNet kernels not available — PyTorch fallback will be used (slower but works).')

# Step 3: If P100, replace torch with cu118 build that has sm_60 kernels
if P100_MODE:
    print('P100 — installing torch 2.7.1+cu118 (sm_60 support)...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.7.1+cu118',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
        '--force-reinstall', '--no-deps',
    ])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'nvidia-cuda-cupti-cu11',
        'nvidia-cuda-runtime-cu11',
        'nvidia-cuda-nvrtc-cu11',
        'nvidia-cublas-cu11',
        'nvidia-cufft-cu11',
        'nvidia-curand-cu11',
        'nvidia-cusolver-cu11',
        'nvidia-cusparse-cu11',
        'nvidia-nccl-cu11',
        'nvidia-nvtx-cu11',
        'nvidia-cudnn-cu11==9.1.0.70',
    ])
    stale = [k for k in sys.modules if k == 'torch' or k.startswith('torch.')]
    for k in stale:
        del sys.modules[k]

# Verify torch + transformers
check = subprocess.run([sys.executable, '-c',
    'import torch; print(f"torch={torch.__version__}"); print(f"archs={torch.cuda.get_arch_list()}"); '
    'import transformers; print(f"transformers={transformers.__version__}")'],
    capture_output=True, text=True)
print(f'Versions: {check.stdout.strip()}')
if check.returncode != 0:
    print(f'ERROR: {check.stderr.strip()}')

print('All dependencies installed.')

In [ ]:
# Cell 2: Verify GPU and torch build
import torch
print(f"torch version: {torch.__version__}")
print(f"torch CUDA archs: {torch.cuda.get_arch_list()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    vram = gpu.total_mem if hasattr(gpu, 'total_mem') else gpu.total_memory
    print(f"VRAM: {vram / 1024**3:.1f} GB")
    print(f"Compute capability: {gpu.major}.{gpu.minor}")
    COMPUTE_CAP = gpu.major * 10 + gpu.minor
    USE_4BIT = COMPUTE_CAP >= 70
    if not USE_4BIT:
        print(f"\nWARNING: Compute capability {gpu.major}.{gpu.minor} < 7.0")
        print("bitsandbytes CUDA kernels not supported on this GPU.")
        print("Falling back to fp16 (no 4-bit quantization).")
        if 'sm_60' in torch.cuda.get_arch_list():
            print("torch has sm_60 support — CUDA ops should work.")
        else:
            raise RuntimeError("torch lacks sm_60 — cannot run on P100!")
    else:
        print(f"4-bit quantization supported (sm_{COMPUTE_CAP}).")
else:
    raise RuntimeError("No GPU! Enable GPU in Settings > Accelerator.")

In [ ]:
# Cell 3: Check training data
import os, json

# Kaggle datasets are mounted at /kaggle/input/
DATA_PATHS = [
    '/kaggle/input/attestor-43-training-data/training_data_v3.jsonl',
    '/kaggle/input/attestor-training-data/training_data_v3.jsonl',
    '/kaggle/input/training_data_v3.jsonl',
    '/kaggle/working/training_data_v3.jsonl',
    '/kaggle/input/attestor-43-training-data/training_data_merged.jsonl',
    '/kaggle/input/attestor-training-data/training_data_merged.jsonl',
    '/kaggle/input/training_data_merged.jsonl',
    '/kaggle/working/training_data_merged.jsonl',
]

DATA_FILE = None
for p in DATA_PATHS:
    if os.path.exists(p):
        DATA_FILE = p
        break

if DATA_FILE is None:
    print('Training data not found at expected paths. Searching...')
    for root, dirs, files in os.walk('/kaggle/input'):
        for fname in ('training_data_v3.jsonl', 'training_data_merged.jsonl'):
            if fname in files:
                DATA_FILE = os.path.join(root, fname)
                print(f'Found: {DATA_FILE}')
                break
        if DATA_FILE:
            break
    if DATA_FILE is None:
        raise FileNotFoundError(
            'training_data_v3.jsonl not found! Upload it as a Kaggle dataset.\n'
            'Expected at: /kaggle/input/attestor-43-training-data/training_data_v3.jsonl')

with open(DATA_FILE, 'r', encoding='utf-8') as f:
    examples = [json.loads(line) for line in f if line.strip()]
print(f'Loaded {len(examples)} training examples from {DATA_FILE}')
print(f'File size: {os.path.getsize(DATA_FILE) / 1024 / 1024:.1f} MB')

# Sanity check — must have 3000+ examples
if len(examples) < 1000:
    raise ValueError(f'Only {len(examples)} examples loaded — wrong file? Expected 3700+.')

# Report SSTI/SSRF ratio for verification
ssti = sum(1 for e in examples if 'ssti' in (e.get('instruction','')+e.get('output','')).lower() or 'cwe-94' in (e.get('instruction','')+e.get('output','')).lower())
ssrf = sum(1 for e in examples if 'cwe-918' in (e.get('instruction','')+e.get('output','')).lower())
print(f'SSTI/CWE-94 examples: {ssti}, SSRF/CWE-918 examples: {ssrf}')

print(f'Sample: {json.dumps(examples[0], indent=2)[:300]}')

In [ ]:
# Cell 4: Configuration — model selection based on GPU
import os, torch
OUTPUT_DIR = '/kaggle/working/attestor-43-lora'

# Reduce CUDA fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

NUM_GPUS = torch.cuda.device_count()
print(f'GPUs available: {NUM_GPUS}')
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    vram = props.total_mem if hasattr(props, 'total_mem') else props.total_memory
    print(f'  GPU {i}: {props.name} — {vram / 1024**3:.1f} GiB')

if USE_4BIT:
    MODEL_NAME = 'empero-ai/Qwythos-9B-Claude-Mythos-5-1M'
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
    OPTIM = 'paged_adamw_8bit'
    LORA_R = 16
    LORA_ALPHA = 32
    MAX_SEQ_LENGTH = 512 if NUM_GPUS >= 2 else 256
    LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
    print(f'Mode: QLoRA on Qwythos-9B (Mythos-class, 4-bit, batch=1)')
else:
    MODEL_NAME = 'cognitivecomputations/dolphin-2.9.3-mistral-7B-32k'
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
    OPTIM = 'adamw_torch'
    LORA_R = 16
    LORA_ALPHA = 32
    MAX_SEQ_LENGTH = 512
    LORA_TARGET_MODULES = ['q_proj', 'v_proj']
    print(f'P100: Using Dolphin Mistral 7B fallback')

IS_QWYTHOS = 'Qwythos' in MODEL_NAME

LORA_DROPOUT = 0.05

# v3: 1 epoch — v2 used 3 epochs which caused overfitting (loss 1.39, terse memorized outputs)
EPOCHS = 1
LR = 2e-4
SAVE_STEPS = 200
LOGGING_STEPS = 25
USE_FP16 = True
USE_BF16 = False

print(f'Model: {MODEL_NAME}')
print(f'LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}, targets: {LORA_TARGET_MODULES}')
print(f'Epochs: {EPOCHS}, batch: {BATCH_SIZE}, grad_accum: {GRAD_ACCUM}')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')
print(f'Max seq length: {MAX_SEQ_LENGTH}')
print(f'Optimizer: {OPTIM}')

In [ ]:
# Cell 5: Load model (4-bit QLoRA on T4+, fp16 LoRA on P100)

# Fix: BloomPreTrainedModel removed in latest transformers but still imported by peft.
# Use a bare stub — don't inherit from PreTrainedModel to avoid cascading import issues.
try:
    from transformers import BloomPreTrainedModel
except Exception:
    import transformers
    transformers.BloomPreTrainedModel = type('BloomPreTrainedModel', (), {})
    print('Patched BloomPreTrainedModel stub.')

from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def load_model(model_name, quant_config=None, dtype=None):
    kwargs = dict(device_map='auto', trust_remote_code=True)
    if quant_config:
        kwargs['quantization_config'] = quant_config
    if dtype:
        kwargs['torch_dtype'] = dtype
    try:
        from transformers import AutoModelForCausalLM
        return AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    except (ValueError, KeyError, TypeError) as e:
        print(f'CausalLM load failed ({e}), trying ImageTextToText...')
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText.from_pretrained(model_name, **kwargs)

if USE_4BIT:
    print('Using 4-bit quantization (QLoRA)')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = load_model(MODEL_NAME, quant_config=bnb_config)
    # Skip prepare_model_for_kbit_training — its fp32 upcast OOMs on T4 with 9B model.
    # Manually enable what's needed for QLoRA training instead.
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
else:
    print('Using fp16 (no quantization — bitsandbytes not supported on this GPU)')
    model = load_model(MODEL_NAME, dtype=torch.float16)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

print('Base model loaded.')
print(f'Model parameters: {model.num_parameters():,}')

In [ ]:
# Cell 6: Apply LoRA adapter
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
trainable, total = model.num_parameters(only_trainable=True), model.num_parameters()
print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 7: Prepare dataset — ChatML format, pre-tokenized with truncation
from datasets import Dataset

SYSTEM_MSG = (
    'You are Owen Coder, a Mythos-class security-focused code analysis model trained on '
    'the Attestor static analysis engine. You detect vulnerabilities, write exploit PoCs, '
    'perform taint analysis, generate hardening recommendations, and write production-quality '
    'code. You reason step by step inside <think> tags before answering. Be precise, '
    'technical, and exhaustive. Reference exact line numbers and CWE IDs. No refusals.'
)

def tokenize_example(example):
    text = (
        '<|im_start|>system\n'
        f'{SYSTEM_MSG}<|im_end|>\n'
        f'<|im_start|>user\n{example["instruction"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{example["output"]}<|im_end|>'
    )
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
        return_attention_mask=True,
    )
    # Don't set labels here — DataCollatorForLanguageModeling creates them
    # after padding, and sets -100 on padding positions automatically.
    return tokenized

dataset = Dataset.from_list(examples)
dataset = dataset.map(tokenize_example, remove_columns=dataset.column_names)

lengths = [len(x['input_ids']) for x in dataset]
print(f'Dataset: {len(dataset)} examples')
print(f'Token lengths — min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)/len(lengths):.0f}')
print(f'Truncated to {MAX_SEQ_LENGTH}: {sum(1 for l in lengths if l == MAX_SEQ_LENGTH)} examples')
print(f'Base model: {"Qwythos-9B (Mythos-class)" if IS_QWYTHOS else "Dolphin Mistral 7B"}')

In [ ]:
# Cell 8: Train using regular Trainer (not SFTTrainer)
import os, re, time, traceback
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

os.makedirs(OUTPUT_DIR, exist_ok=True)

all_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    fp16=USE_FP16,
    bf16=USE_BF16,
    tf32=False,
    optim=OPTIM,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    report_to='none',
    gradient_checkpointing=True,
    max_grad_norm=0.3,
)

# Iteratively remove kwargs that TrainingArguments rejects
config_kwargs = dict(all_kwargs)
rejected = []
for _ in range(len(config_kwargs)):
    try:
        training_args = TrainingArguments(**config_kwargs)
        break
    except TypeError as e:
        m = re.search(r"unexpected keyword argument '(\w+)'", str(e))
        if m:
            bad = m.group(1)
            rejected.append(bad)
            config_kwargs.pop(bad, None)
        else:
            raise
else:
    raise RuntimeError(f'Could not create TrainingArguments after removing {rejected}')

if rejected:
    print(f'TrainingArguments rejected (removed): {rejected}')

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

# === DRY RUN: one batch forward+backward before committing to full training ===
import gc, torch
gc.collect()
torch.cuda.empty_cache()

print('=== DRY RUN: testing one batch ===')
try:
    # Grab a batch the same way the trainer does
    dl = trainer.get_train_dataloader()
    batch = next(iter(dl))
    print(f'Batch keys: {list(batch.keys())}')
    for k, v in batch.items():
        if hasattr(v, 'shape'):
            print(f'  {k}: shape={v.shape}, dtype={v.dtype}')

    # Move to device
    batch = {k: v.to(trainer.model.device) if hasattr(v, 'to') else v for k, v in batch.items()}

    # Forward
    t0 = time.time()
    model.train()
    with torch.amp.autocast('cuda', dtype=torch.float16):
        outputs = model(**batch)
    loss = outputs.loss
    t_fwd = time.time() - t0
    print(f'Forward OK — loss={loss.item():.4f}, time={t_fwd:.1f}s')

    # Backward
    t0 = time.time()
    loss.backward()
    t_bwd = time.time() - t0
    print(f'Backward OK — time={t_bwd:.1f}s')

    # Memory after one step
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            free = torch.cuda.mem_get_info(i)[0] / 1024**3
            total = torch.cuda.mem_get_info(i)[1] / 1024**3
            print(f'GPU {i}: {free:.1f} GiB free / {total:.1f} GiB total')

    # Time estimate
    step_time = t_fwd + t_bwd
    total_steps = len(dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS
    est_hours = (step_time * total_steps * GRAD_ACCUM) / 3600
    print(f'\nEstimated training time: {step_time:.1f}s/sample × {total_steps * GRAD_ACCUM} samples = {est_hours:.1f} hours')

    # Clear gradients from dry run
    model.zero_grad()
    gc.collect()
    torch.cuda.empty_cache()
    print('=== DRY RUN PASSED — starting full training ===\n')

except Exception as e:
    print(f'\n=== DRY RUN FAILED ===')
    traceback.print_exc()
    raise

# === Full training ===
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free = torch.cuda.mem_get_info(i)[0] / 1024**3
        total = torch.cuda.mem_get_info(i)[1] / 1024**3
        print(f'GPU {i}: {free:.1f} GiB free / {total:.1f} GiB total')

total_steps = len(dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS
print(f'Starting training: {len(dataset)} examples, {EPOCHS} epoch(s)')
print(f'Total optimizer steps: ~{total_steps}')
print(f'Checkpoints every {SAVE_STEPS} steps to {OUTPUT_DIR}')
print(f'Using base Trainer (no SFTTrainer / chunked CE)')
print('---')

trainer.train()
print('\n--- Training complete! ---')

In [ ]:
# Cell 9: Save final adapter
FINAL_DIR = '/kaggle/working/attestor-43-lora-final'
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'LoRA adapter saved to {FINAL_DIR}')
print('Files:')
for f in os.listdir(FINAL_DIR):
    size = os.path.getsize(os.path.join(FINAL_DIR, f))
    print(f'  {f}: {size/1024/1024:.1f} MB')

In [ ]:
# Cell 10: Package for download
import shutil
shutil.make_archive('/kaggle/working/attestor-43-lora', 'zip', FINAL_DIR)
print('Packaged as /kaggle/working/attestor-43-lora.zip')
print(f'Size: {os.path.getsize("/kaggle/working/attestor-43-lora.zip") / 1024 / 1024:.1f} MB')
print('\nDownload from the Output tab on the right, or use the Kaggle API.')